# Homework 4: Evaluation Metrics for Classification

Machine Learning Zoomcamp 2026 — Module 4

Dataset: `course_lead_scoring_2026.csv`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../datasets/course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

Fill categorical missing values with `'NA'`, numerical with `0.0`.

In [ ]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
features = categorical + numerical

for c in categorical:
    df[c] = df[c].fillna('NA')
for c in numerical:
    df[c] = df[c].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Split the data

Exact calls given in the homework (`random_state=1`, different from Homework 3's 42).

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Q1. ROC AUC feature importance

Use each numerical column directly as a prediction score against `converted` on the training set. Invert (negate) any column whose AUC comes out below 0.5.

In [ ]:
for col in ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']:
    auc = roc_auc_score(y_train, df_train[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[col])
    print(f'{col}: {auc:.4f}')

lead_score: 0.7882
number_of_courses_viewed: 0.7230
interaction_count: 0.7649
annual_income: 0.6082


## Q2. Training the model

One-hot encode with `DictVectorizer`, train, report validation AUC.

In [ ]:
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_val)[:, 1]
round(roc_auc_score(y_val, y_pred_proba), 3)

0.732

## Q3. Precision and recall

Sweep thresholds 0.00 to 1.00 in steps of 0.01. Find where precision and recall are closest (ignoring thresholds where both are zero).

In [ ]:
thresholds = np.arange(0.0, 1.01, 0.01)
precisions, recalls = [], []

for t in thresholds:
    y_pred_t = (y_pred_proba >= t).astype(int)
    tp = ((y_pred_t == 1) & (y_val == 1)).sum()
    fp = ((y_pred_t == 1) & (y_val == 0)).sum()
    fn = ((y_pred_t == 0) & (y_val == 1)).sum()
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    precisions.append(p)
    recalls.append(r)

precisions = np.array(precisions)
recalls = np.array(recalls)
diff = np.abs(precisions - recalls)
valid = ~((precisions == 0) & (recalls == 0))
best_idx = np.argmin(np.where(valid, diff, np.inf))
round(thresholds[best_idx], 2)

np.float64(0.63)

## Q4. F1 score

At which threshold is F1 maximal?

In [ ]:
f1s = []
for p, r in zip(precisions, recalls):
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    f1s.append(f1)

f1s = np.array(f1s)
round(thresholds[np.argmax(f1s)], 2)

np.float64(0.41)

## Q5. 5-fold cross-validation

`KFold(n_splits=5, shuffle=True, random_state=1)` over `df_full_train`, `C=1.0`. What's the standard deviation of AUC across folds?

In [ ]:
def train_eval(df_train_fold, df_val_fold, C):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train_fold[features].to_dict(orient='records'))
    X_val = dv.transform(df_val_fold[features].to_dict(orient='records'))
    y_train_fold = df_train_fold['converted'].values
    y_val_fold = df_val_fold['converted'].values

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    model.fit(X_train, y_train_fold)
    y_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val_fold, y_pred)

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
scores = []
for train_idx, val_idx in kfold.split(df_full_train):
    df_train_fold = df_full_train.iloc[train_idx]
    df_val_fold = df_full_train.iloc[val_idx]
    scores.append(train_eval(df_train_fold, df_val_fold, C=1.0))

scores, round(np.std(scores), 3)

([0.7305601027122453,
  0.7295726148196622,
  0.7363428071259397,
  0.7412454717957564,
  0.7209138590326581],
 np.float64(0.007))

## Q6. Hyperparameter tuning

Same 5-fold CV, sweep `C`, pick the best mean AUC (ties broken by lowest std, then smallest `C`).

In [ ]:
results = {}
for C in [0.000001, 0.001, 1]:
    kfold = KFold(n_splits=5, shuffle=True, random_state=1)
    scores_c = []
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train_fold = df_full_train.iloc[train_idx]
        df_val_fold = df_full_train.iloc[val_idx]
        scores_c.append(train_eval(df_train_fold, df_val_fold, C=C))
    results[C] = (round(np.mean(scores_c), 3), round(np.std(scores_c), 3))

results

{1e-06: (np.float64(0.617), np.float64(0.019)),
 0.001: (np.float64(0.749), np.float64(0.01)),
 1: (np.float64(0.732), np.float64(0.007))}

In [ ]:
best_C = max(results, key=lambda c: (results[c][0], -results[c][1], -c))
best_C

0.001